# Scoring and audit, part 2: HT-3 final scores and HT-5

Decisions D280 to D289 (DESIGN_ADDENDUM P). Script: `scripts/scoring_audit_p2.py`. It imports
`scripts/scoring_audit.py` (part 1) for small helpers, and `scoring`, `normalise` and `stats` from the
detached checkout of the tag gate-E (D271, D283).

**Runtime: CPU.** No API call, no GPU, no cost. Run the cells in order. Every cell prints ids and counts
only. Files that hold text stay in Drive (`TEXT_*.csv`, the sheets, the packages).

**People (D281).** One runner runs this notebook and prepares the packages; the runner is never an auditor.
Two auditors, different people, neither of them the runner. The auditors never see this notebook's outputs,
and they do not open Drive `scored/`, `results/`, `human/HT-3/` or `human/HT-5/` until both sheets are
returned.

**Before Cell 0:** `scripts/scoring_audit_p2.py` and this notebook are committed on `main` of
`banglaindoorvqa-main` (line C07 fails otherwise). Commit the notebook with its outputs cleared. Upload
`HT5_auditor_guide.md` and `HT5_sensitivity_instructions.md` to Drive `human/HT-5/guide/`.

Do **not** run Cell 9 of `asset12_ht3.ipynb`. Cell 2 here runs the same command (`score --final`) from the
gate-F checkout (D283). On a new runtime, run Cells P and 0 again before any later cell.

## Cell P: settings
Set `REPO_SLUG` to the owner of `banglaindoorvqa-main` (as in `scoring.ipynb`). Fill the auditor lines only
before Cell 4.

In [ ]:
# Cell P: settings
DRIVE_ROOT = "/content/drive/MyDrive/BanglaIndoorVQA"   # the Drive project folder
REPO_SLUG = "OWNER/banglaindoorvqa-main"                 # set OWNER
MAIN = "/content/banglaindoorvqa-main"                   # the main clone
GATE_E = "/content/bivqa_gateE"                          # detached checkout of the tag gate-E
GATE_F = "/content/bivqa_gateF"                          # detached checkout of the tag gate-F
SCRIPT = MAIN + "/scripts/scoring_audit_p2.py"
# Cell 2 only: leave empty. A second HT-3 final score needs a written reason and a decision row.
HT3_AGAIN = ""
# Cell 4 only (D281). Earlier roles of each auditor, comma separated, from:
# none, HT-1, HT-3 N1, HT-3 N2, HT-4, HT-6. Roles only, never names.
AU1_ROLES = ""
AU2_ROLES = ""
AUDITORS_CLEARED = False   # True only after both auditors confirmed the rules (guide, section 2)

## Cell 0: clone, pull, the two checkouts, input check
Mounts Drive, clones `banglaindoorvqa-main` with the Colab secret `GITHUB_TOKEN` (never printed, never
written to `.git/config`), pulls, fetches the tags, makes the detached checkouts of gate-E and gate-F, then
runs the read-only check.

**Correct output:** the Git steps end with `SETUP: PASS`. Then lines C01 to C26, each `PASS` or `INFO`, and
`CHECK: ALL PASS (26 lines)`. Look for: C02 `gate-E tag on 5db2bfff3fd3, gate-F tag on a5164e4a0b82, tag
object c5de990d524c`; C11 `28 rows`; C13 the six scored files equal the gate-F tag message; C14 `0 FAIL;
51 model-condition sets`; C18 `53 rows, 48 item and sibling pairs`; C20 `changes.csv 304 rows (rule a 294,
rule b 10)`; C21 `4406 rows`. C23 says `not yet (Cell 3)` and C24 says the guide is present once uploaded.

**Send back:** the whole printout of this cell.

In [ ]:
# Cell 0: setup and the input check
import os, subprocess, sys
from google.colab import drive, userdata

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"      # no __pycache__ in a Git tree (D252)
sys.dont_write_bytecode = True
drive.mount("/content/drive")

if "OWNER" in REPO_SLUG:
    raise SystemExit("FAIL: set REPO_SLUG in Cell P")
if not REPO_SLUG.endswith("/banglaindoorvqa-main"):
    raise SystemExit("FAIL: REPO_SLUG must name banglaindoorvqa-main, not the dataset repo")
try:
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise SystemExit("FAIL: the Colab secret GITHUB_TOKEN is missing or not allowed for this "
                     "notebook (%s)" % type(exc).__name__)
if not TOKEN:
    raise SystemExit("FAIL: the Colab secret GITHUB_TOKEN is empty")
PLAIN = "https://github.com/%s.git" % REPO_SLUG
AUTHED = "https://x-access-token:%s@github.com/%s.git" % (TOKEN, REPO_SLUG)


def sh(*args):
    p = subprocess.run(list(args), capture_output=True, text=True)
    shown = " ".join(args).replace(TOKEN, "***")
    out = (p.stdout + p.stderr).replace(TOKEN, "***").strip()
    print("%s  %s%s" % ("PASS" if p.returncode == 0 else "FAIL", shown, ("\n" + out) if out else ""))
    if p.returncode:
        raise SystemExit("SETUP: FAIL (exit %d)" % p.returncode)
    return out


if not os.path.isdir(MAIN + "/.git"):
    sh("git", "clone", "-q", AUTHED, MAIN)
sh("git", "-C", MAIN, "remote", "set-url", "origin", PLAIN)       # no token in .git/config
sh("git", "-C", MAIN, "checkout", "-q", "main")
sh("git", "-C", MAIN, "fetch", "-q", AUTHED,
   "+refs/heads/main:refs/remotes/origin/main", "+refs/tags/*:refs/tags/*")
sh("git", "-C", MAIN, "merge", "-q", "--ff-only", "origin/main")   # the pull
sh("git", "-C", MAIN, "worktree", "prune")
for path, tag in ((GATE_E, "gate-E"), (GATE_F, "gate-F")):
    if not os.path.isdir(path):
        sh("git", "-C", MAIN, "worktree", "add", "-q", "--detach", path, tag)
    print("%s checkout HEAD %s" % (tag, sh("git", "-C", path, "rev-parse", "--short=12", "HEAD").splitlines()[-1]))
print("main HEAD", sh("git", "-C", MAIN, "rev-parse", "--short=12", "HEAD").splitlines()[-1])
print("SETUP: PASS")


def run(*args):
    cmd = [sys.executable, "-u", SCRIPT, "--root", DRIVE_ROOT, "--gate-e", GATE_E, "--gate-f", GATE_F,
           "--main", MAIN] + list(args)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"))
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    print("exit code %d" % rc)
    return rc


run("check")

## Cell 1: selftest on made-up answers
Builds a small made-up project in a temporary folder and runs the HT-5 chain with the gate-E scorer and
stats.py: draw, two made-up auditors, receipts, consensus, the D238 file, the reading, the sensitivity
template and check, the locks. Reads no project data.

**Correct output:** `SELFTEST: 35 of 35 checks PASS`. **Send back:** that line (the whole printout if any
line says FAIL).

In [ ]:
# Cell 1: selftest
run("selftest")

## Cell 2: HT-3 final scores (D208, D283)
Runs `asset12_nameref.py check-inputs` and then `score --final` from the gate-F checkout (the scorer and
normalise of gate-F, the lexicon of gate-A, the 28-row excluded list), then reads the new folder as the
Statistics chat will (`stats.read_ht3`). It never scores a second time unless `HT3_AGAIN` holds a reason.

**Correct output:** `check-inputs: ALL PASS`; the HT-3 report with `FINAL (gate-F scorer, gate-A lexicon)`
and `Items: 200 drawn, 1 excluded at scoring, 199 counted, 0 unscoreable`; `Wrote
.../human/HT-3/score_final_<UTC>`; then lines T01 to T10, with T07 `items drawn 200, counted 199, excluded at
scoring 1 (H54_0006_q1), unscoreable 0; name rows 400 (N1 200, N2 200); primary role N1, lexicon writer N2`
and T08 `PASS`; then `HT-3 FINAL: PASS`.

**Send back:** the whole printout, and three Drive files: `human/HT-3/score_final_<UTC>/ht3_report.txt`,
`human/HT-3/score_final_<UTC>/ht3_same_other_object.csv` and
`human/HT-3/final_check/TEXT_ht3_final_sample10.csv` (the 10-item check). Wait for the reply before Cell 4.

In [ ]:
# Cell 2: ht3-final
run(*(["ht3-final"] + (["--again", HT3_AGAIN] if HT3_AGAIN else [])))

## Cell 3: the HT-5 frame (D234, D283)
Runs `analysis.py budget-items --mode real` from the gate-E checkout on the six scored files, then checks
the file it wrote against `stats.error_budget`. Refuses once an HT-5 draw exists (the frame never changes
after the draw).

**Correct output:** the analysis.py check lines (all `PASS`, including the lines `... at tag gate-E:
identical`), three lines `budget-items <model>: N items not at Tier 1 in N-BB; A1 .., A2 .., A3 .., A4 ..,
B .., C ..`, `Wrote .../results/stats_inputs/error_budget_items.csv`, then B01 to B07 with B01, B05, B06 and
B07 `PASS`, and `BUDGET FRAME: PASS`.

**Send back:** the whole printout. Wait for the reply before Cell 4.

In [ ]:
# Cell 3: budget-items
run("budget-items")

## Before Cell 4: the auditors
Two auditors, different people, neither of them the runner (D281). Each confirms, before the draw, that
they will work alone without AI tools or web search, and will not open Drive `scored/`, `results/`,
`human/HT-3/` or `human/HT-5/` (other than the files they are sent) until both sheets are returned. Write
their earlier roles in Cell P (roles only, never names) and set `AUDITORS_CLEARED = True`.

## Cell 4: the HT-5 draw (D280 to D286)
Draws the 200 answers (140 C, 30 English copies, 30 siblings), adds the D192 answers and one answer per
HT-3 look-first item beside them, checks every picture against `images_manifest.csv`, writes one package
per auditor (sheet, picture page, guide, pictures) with its own order and codes, and the key. A second run
changes nothing.

**Correct output:** `Sample (D284): 200 answers: C 140, english_copy 30, sibling 30` (or a stated
shortfall); the D192 line `touched and scored wrong d192_1 2, d192_2 0, d192_3 0, d192_4 2, d192_5 0, d192_6
6, d192_7 4, 14 answers (equal to addendum O2)`; the HT-3 line; `Rows per auditor:` about 210 to 235;
`Leak scan: PASS`; two package lines (each about 20 to 60 MB); `HT-5 DRAW: DONE`.

**Send back:** the whole printout, and wait for the reply. Then send `human/HT-5/to_send/HT5_AU1.zip` to
auditor 1 and `HT5_AU2.zip` to auditor 2. Send nothing else, and never the `key/` folder.

In [ ]:
# Cell 4: ht5-draw
if not AUDITORS_CLEARED:
    raise SystemExit("Not drawn. Set AU1_ROLES, AU2_ROLES and AUDITORS_CLEARED = True in Cell P first.")
run("ht5-draw", "--auditors-cleared", "--au1-roles", AU1_ROLES, "--au2-roles", AU2_ROLES)

## Cell 5: receive the returned sheets
Save each returned file, unchanged, as `human/HT-5/received/HT5_sheet_AU1_returned.xlsx` and
`HT5_sheet_AU2_returned.xlsx`, then run this cell. It may run as often as needed; a receipt never changes.
A sheet with problems is not recorded: the cell lists the codes to fix; send that list to the auditor.

**Correct output:** `AU1: PASS ...`, `AU2: PASS ...`, `RECEIPT: 2 of 2`, two agreement lines, and the path
of `to_send/HT5_consensus.zip` (or `No row differs`). Give the consensus package to both auditors together.

**Send back:** the printout once it says `RECEIPT: 2 of 2`.

In [ ]:
# Cell 5: ht5-receive
run("ht5-receive")

## Cell 6: the consensus and the D238 file
Save the filled consensus sheet as `human/HT-5/received/HT5_consensus_returned.xlsx` (skip this if no row
differed), then run this cell. Writes `human/HT-5/ht5_audit.csv` (D238: the 200 sample answers),
`ht5_verdicts.csv` (every row, ids and both raw verdicts), `TEXT_ht5_forms.csv` (the agreed missing names)
and `TEXT_ht5_notes.csv`.

**Correct output:** `Consensus sheet: n rows settled`, the four files, `CONSENSUS: DONE`. **Send back:** the
printout.

In [ ]:
# Cell 6: ht5-consensus
run("ht5-consensus")

## Cell 7: the HT-5 reading
Reads `ht5_audit.csv` with `stats.read_ht5` and `stats.ht5_stats` (gate-E copy), as the Statistics chat
will, and prints the counts this chat reports: auditor agreement, verdicts by model and scorer label, D204,
the HT-3 look-first rows, D279, the D192 rows and the missing-variant list.

**Correct output:** R01 to R03 `PASS` (R03 shows `0 FAIL` and no strata warning when the sample is 140, 30
and 30), R04 to R14 `INFO`, `HT-5 READING: PASS`. **Send back:** the whole printout.

In [ ]:
# Cell 7: ht5-read
run("ht5-read")

## Cell 8: the lexicon sensitivity template (D238, D287)
Writes the empty `lexicon_sensitivity_TEMPLATE.csv` (header only) and `HT5_sensitivity_worksheet.xlsx`.
Then the two auditors, together, write the rows by `HT5_sensitivity_instructions.md` and save them as
`human/HT-5/lexicon_sensitivity.csv`. Nobody else and no program writes a row.

**Correct output:** the two file lines and `NEXT: ...`. Nothing to send back.

In [ ]:
# Cell 8: sens-template
run("sens-template")

## Cell 9: check lexicon_sensitivity.csv
Reads the file with `stats.read_sensitivity`, checks that every HT-5 row uses an agreed form, applies the
rows to copies of the frozen files (`stats._apply_rows`) and counts the answers they change
(`stats.sensitivity_records`), exactly as the Statistics chat will. Run it again after any fix.

**Correct output:** X01 to X04 `PASS`, X05 to X08 `INFO`, `SENS-CHECK: PASS`. **Send back:** the whole
printout.

In [ ]:
# Cell 9: sens-check
run("sens-check")

## Cell 10: status
What exists so far. Read-only. **Send back:** the printout at the end, before the chat closes.

In [ ]:
# Cell 10: status
run("status")